# LC 525 - Contiguous Array

## Problem

Given a binary array `nums`, find the maximum length of a contiguous subarray containing an equal number of `0` and `1`.

Example:

`nums = [0,1,0,1]`

The entire array has:
- 2 zeros
- 2 ones

Answer = `4`

---

## Brute Force Approach

### Idea

Check every possible subarray and count:
- number of `0`s
- number of `1`s

If they are equal, calculate its length and keep the maximum.

### Complexity

- Time: `O(n²)` if prefix information is used to count zeros/ones efficiently
- Space: `O(1)` or `O(n)` depending on implementation

This works, but we can do better.

---

# Optimal Approach

## Pattern

**Prefix Sum + Hash Map + Equal State**

This is a variation of the **Prefix Sum + Hash Map** pattern used in:

- LC 560 - Subarray Sum Equals K
- LC 525 - Contiguous Array

The key difference is how we represent the array.

---

## Key Observation

We need:

> Equal number of `0`s and `1`s.

Convert the problem into a balance problem:

- Treat `1` as `+1`
- Treat `0` as `-1`

Then:

`equal number of 0s and 1s`

becomes:

`sum = 0`

Example:

`[0,1,1,0]`

Becomes:

`[-1,+1,+1,-1]`

Prefix sums:

| Index | Value | Prefix Sum |
|---:|---:|---:|
| 0 | -1 | -1 |
| 1 | +1 | 0 |
| 2 | +1 | 1 |
| 3 | -1 | 0 |

The prefix sum `0` appears at index `1` and again at index `3`.

Therefore, the subarray between them has sum `0`.

That means it contains an equal number of `0`s and `1`s.

Length:

`3 - 1 = 2`

---

# Why Repeated Prefix Sum Works

Suppose the prefix sum at index `i` is the same as the prefix sum at an earlier index `j`.

Then:

`prefix[i] = prefix[j]`

Therefore:

`prefix[i] - prefix[j] = 0`

So the elements between `j+1` and `i` have total balance `0`.

Since:

- `1 = +1`
- `0 = -1`

balance `0` means:

`number of 1s = number of 0s`

Therefore, we found a valid subarray.

---

# Why Store the First Occurrence?

This is extremely important.

Suppose a prefix sum appears at:

`index 2`

and later at:

`index 8`

The length is:

`8 - 2 = 6`

If the same prefix sum also appeared at index `5`, using index `5` would give:

`8 - 5 = 3`

We always want the **longest** subarray.

Therefore:

> Store the earliest index where each prefix sum appears.

That is why your code only does:

`location[prefix_sum] = i`

when the prefix sum is not already present.

---

# Initial State

Your map starts with:

`location = {0: -1}`

This represents:

> Before the array starts, the prefix sum is `0` at index `-1`.

This handles subarrays that start from index `0`.

Example:

`[0,1]`

Transformed:

`[-1,+1]`

Prefix sums:

`-1 → 0`

At index `1`, prefix sum `0` already exists at `-1`.

Length:

`1 - (-1) = 2`

So the entire array is correctly counted.

---

# Algorithm

1. Start `prefix_sum = 0`.
2. Store `{0: -1}` in the hash map.
3. Traverse the array.
4. For every `1`, add `1` to the prefix sum.
5. For every `0`, subtract `1`.
6. If the current prefix sum already exists:
   - Calculate the distance from its first occurrence.
   - Update the maximum length.
7. Otherwise, store the current prefix sum and index.
8. Return the maximum length.

---

# Dry Run

### Input

`nums = [0,1,0,1]`

Initial:

`prefix_sum = 0`

`location = {0:-1}`

`max_length = 0`

### Index 0

Value = `0`

So:

`prefix_sum = -1`

`-1` is not in the map.

Store:

`location = {0:-1, -1:0}`

---

### Index 1

Value = `1`

So:

`prefix_sum = 0`

`0` is already in the map at index `-1`.

Length:

`1 - (-1) = 2`

So:

`max_length = 2`

---

### Index 2

Value = `0`

`prefix_sum = -1`

`-1` already exists at index `0`.

Length:

`2 - 0 = 2`

`max_length` remains `2`.

---

### Index 3

Value = `1`

`prefix_sum = 0`

`0` already exists at index `-1`.

Length:

`3 - (-1) = 4`

Therefore:

`max_length = 4`

Answer:

`4`

---

# Primary Pattern

## Prefix Sum + Earliest Index Hash Map

The general structure is:

`state → first index`

During traversal:

1. Update the state.
2. If state has appeared before:
   - current index - earliest index = valid subarray length
3. Otherwise:
   - store the state.

Here, the state is the **balance between 1s and 0s**.

---

# Pattern Recognition

When you see a problem asking for:

- longest subarray
- contiguous subarray
- equal number of two values
- sum/balance becomes zero
- repeated cumulative state

Immediately ask:

> "Can I represent the condition using a prefix state?"

Then ask:

> "If the same state appears twice, does the section between them become valid?"

If yes:

**Prefix State + Hash Map**

---

# Important Connection to LC 560

### LC 560 - Subarray Sum Equals K

There:

`previous_prefix = current_prefix - k`

We search for a specific previous prefix.

### LC 525 - Contiguous Array

Here:

`required sum = 0`

So instead of searching for a specific numerical target, we look for:

> The same prefix balance appearing again.

This is the same larger pattern:

**Prefix State + Hash Map**

---

# Important Difference from LC 485

### LC 485 - Max Consecutive Ones

Question:

> "How long is the current consecutive streak?"

Pattern:

**Running Streak + Reset**

### LC 525 - Contiguous Array

Question:

> "How far apart are two equal prefix balances?"

Pattern:

**Prefix State + Earliest Index**

So although both involve binary arrays, their patterns are completely different.

---

# Common Mistakes

### 1. Using only a running count

A running balance alone cannot tell us the longest valid subarray.

We need to remember where each balance first appeared.

---

### 2. Overwriting an existing index

Wrong idea:

`location[prefix_sum] = i`

every time.

This destroys the earliest position and can produce a shorter answer.

Correct idea:

> Store only the first occurrence.

---

### 3. Forgetting `{0:-1}`

Without:

`location = {0:-1}`

you can miss valid subarrays that start at index `0`.

---

### 4. Treating 0 and 1 the same

They need opposite contributions:

`1 → +1`

`0 → -1`

because we want their counts to cancel.

---

### 5. Using a sliding window automatically

A standard sliding window is not suitable here because the array can contain patterns where expanding/shrinking does not give a monotonic condition.

The prefix-state approach works regardless of the arrangement of `0`s and `1`s.

---

# Complexity

Let `n = len(nums)`.

### Time

`O(n)`

Every element is processed once.

Hash-map operations are `O(1)` average.

### Space

`O(n)`

In the worst case, we may store up to `n` different prefix balances.

---

# Interview Thinking

If asked:

### "Why does the repeated prefix sum prove equal zeros and ones?"

Answer:

> I represent `1` as `+1` and `0` as `-1`. If the same prefix sum occurs at two indices, the difference between those prefix sums is zero. Therefore, the subarray between them has equal positive and negative contributions, meaning equal numbers of `1`s and `0`s.

### "Why store the first occurrence?"

Answer:

> For the same prefix balance, the earliest index produces the longest possible subarray ending at the current index.

### "Why initialize with `{0:-1}`?"

Answer:

> It represents a zero balance before the array begins, allowing us to correctly calculate subarrays that start at index `0`.

---

# Generalization

This pattern extends beyond binary arrays.

Whenever a problem says:

> "Find the longest subarray where some cumulative condition becomes equal/repeated"

try:

**Convert condition → Prefix State → Hash Map → Earliest Occurrence**

Examples:

- Equal 0s and 1s → balance
- Subarray sum = K → prefix sum difference
- Equal counts of multiple categories → multi-dimensional/state encoding
- Longest subarray with a repeated cumulative state → prefix state + earliest index

---

# Pattern Hierarchy

Your array problems are now connecting into a larger structure:

`Array Traversal`
↓
`Running State`
↓
`Prefix State`
↓
`Prefix State + Hash Map`
↓
`Longest Valid Subarray`

You should recognize LC 525 as a **harder evolution of simple array traversal**.

---

# Key Takeaway

The entire problem can be reduced to one idea:

> **Turn equal 0s and 1s into a zero-balance problem, then find the farthest-apart repeated prefix balance.**

Mental model:

`0 → -1`

`1 → +1`

`same prefix balance twice → valid subarray`

`earliest occurrence → longest length`

---

# Mastery Checklist

- [x] Understand why `0 → -1` and `1 → +1`
- [x] Understand prefix balance
- [x] Understand why repeated prefix balance creates a valid subarray
- [x] Understand `{0:-1}`
- [x] Understand why only the first occurrence is stored
- [x] Understand hash map role
- [x] Recognize the Prefix State + Hash Map pattern
- [x] Connect LC 525 with LC 560
- [x] Distinguish LC 525 from sliding window
- [x] Analyze `O(n)` time and `O(n)` space
- [x] Handle subarrays starting at index `0`
- [x] Handle all-zero/all-one arrays

## Status: MASTERED

**Primary Pattern:** Prefix Sum / Prefix Balance + Hash Map  
**Core Technique:** Store earliest index of each prefix state  
**Difficulty:** Medium  
**Time:** O(n)  
**Space:** O(n)

In [ ]:
class Solution(object):
    def findMaxLength(self, nums):
        """
        :type nums: List[int]
        :rtype: int
        """
        prefix_sum=0 
        n=len(nums)
        location={0:-1}
        max_length=0
        for i in range(n):
            if nums[i]==1:
                prefix_sum+=1
            else:
                prefix_sum-=1
            if prefix_sum in location:
                length=i-location[prefix_sum]
                max_length=max(max_length,length)
            else:
                location[prefix_sum]=i
        return max_length
        
        